# RS3 Chen2013-only model: fixed split, 100 Optuna trials

Place this notebook in `rs_dev/code` and run all cells.

In [1]:
from pathlib import Path
import json, warnings, joblib
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold
from datasets import dataset_list
from core import get_feature_df

TRACR_FILTER = "Chen2013"
SPLIT_SEED = 42
OPTUNA_SEED = 42
MODEL_SEED = 42
N_TRIALS = 100
EARLY_STOPPING_ROUNDS = 10

PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../models/rs3_chen2013_fixed_split_100_trials")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_NAMES_FILE = PROCESSED_DIR / "train_data_names.csv"
print("Output:", OUTPUT_DIR.resolve())


/opt/anaconda3/envs/rs_dev_venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Output: /Users/zhangjiongyu/CRISPR-modular-deep-learning_backup_copy_revision/Revision_existing_model_comparison/RS3/rs_dev/models/rs3_chen2013_fixed_split_100_trials


In [3]:
train_data_names = pd.read_csv(TRAIN_NAMES_FILE)["name"].dropna().astype(str).tolist()
train_data_list = [ds for ds in dataset_list if ds.name in train_data_names]

for ds in train_data_list:
    ds.load_data()
    ds.set_sgrnas()

sg_df_list = []
for ds in train_data_list:
    df = ds.get_sg_df(include_group=True, include_activity=True).copy()
    df["dataset"] = ds.name
    df["tracr"] = ds.tracr
    sg_df_list.append(df)

groups = (
    pd.concat(sg_df_list, ignore_index=True)
    .groupby("sgRNA Context Sequence", as_index=False)
    .agg(target=("sgRNA Target", lambda x: ", ".join(sorted({
        str(v).upper() for v in x if not pd.isna(v) and str(v).strip() != ""
    }))))
)
groups["target"] = groups.apply(
    lambda r: r["target"] if r["target"] != "" else r["sgRNA Context Sequence"],
    axis=1,
)

all_data = (
    pd.concat(sg_df_list, ignore_index=True)
    .merge(groups[["sgRNA Context Sequence", "target"]], on="sgRNA Context Sequence", how="inner")
    .sort_values(["dataset", "target"])
    .reset_index(drop=True)
)

all_data["sgRNA Activity"] = pd.to_numeric(all_data["sgRNA Activity"], errors="coerce")
all_data = all_data.dropna(subset=[
    "sgRNA Context Sequence", "sgRNA Activity", "dataset", "tracr", "target"
]).reset_index(drop=True)

print("Available tracr values:")
print(all_data["tracr"].value_counts(dropna=False))

filtered_data = all_data.loc[
    all_data["tracr"].astype(str) == TRACR_FILTER
].copy().reset_index(drop=True)

if filtered_data.empty:
    raise ValueError(f"No rows found for tracr={TRACR_FILTER!r}")

print("\nFiltered rows:", len(filtered_data))
display(filtered_data["dataset"].value_counts().rename("n").to_frame())


Available tracr values:
tracr
Hsu2013     29951
Chen2013    21136
Name: count, dtype: int64

Filtered rows: 21136


,n
dataset,
Munoz2016,21136


In [5]:
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
seen_idx, unseen_idx = next(outer.split(
    filtered_data, y=filtered_data["dataset"], groups=filtered_data["target"]
))
seen_data = filtered_data.iloc[seen_idx].reset_index(drop=True)
unseen_data = filtered_data.iloc[unseen_idx].reset_index(drop=True)

inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
train_idx, val_idx = next(inner.split(
    seen_data, y=seen_data["dataset"], groups=seen_data["target"]
))
train_data = seen_data.iloc[train_idx].reset_index(drop=True)
validation_data = seen_data.iloc[val_idx].reset_index(drop=True)

assert set(train_data["target"]).isdisjoint(set(validation_data["target"]))
assert set(train_data["target"]).isdisjoint(set(unseen_data["target"]))
assert set(validation_data["target"]).isdisjoint(set(unseen_data["target"]))

display(pd.DataFrame({
    "subset": ["train", "validation", "unseen"],
    "n_rows": [len(train_data), len(validation_data), len(unseen_data)],
    "fraction": [
        len(train_data)/len(filtered_data),
        len(validation_data)/len(filtered_data),
        len(unseen_data)/len(filtered_data),
    ],
}))


,subset,n_rows,fraction
0,train,12741,0.602810
1,validation,3177,0.150312
2,unseen,5218,0.246877


In [7]:
X_train = get_feature_df(train_data)
X_validation = get_feature_df(validation_data).reindex(columns=X_train.columns, fill_value=0)
X_unseen = get_feature_df(unseen_data).reindex(columns=X_train.columns, fill_value=0)

y_train = train_data["sgRNA Activity"].to_numpy(float)
y_validation = validation_data["sgRNA Activity"].to_numpy(float)
y_unseen = unseen_data["sgRNA Activity"].to_numpy(float)

print(X_train.shape, X_validation.shape, X_unseen.shape)


100%|█████████████████████████████████████| 5218/5218 [00:01<00:00, 4146.15it/s]


(12741, 632) (3177, 632) (5218, 632)


In [9]:
def safe_pearson(y_true, y_pred):
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(pearsonr(y_true, y_pred)[0])

def safe_spearman(y_true, y_pred):
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(y_true, y_pred)[0])


In [11]:
trial_records = []
trial_models = {}

def objective(trial):
    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=MODEL_SEED,
        n_jobs=-1,
        learning_rate=0.01,
        n_estimators=5000,
        num_leaves=trial.suggest_int("num_leaves", 8, 256),
        min_child_samples=trial.suggest_int("min_child_samples", 8, 256),
        verbosity=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_validation, y_validation)],
        eval_metric="mse",
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )

    best_iteration = model.best_iteration_ or model.n_estimators
    val_pred = model.predict(X_validation, num_iteration=best_iteration)
    unseen_pred = model.predict(X_unseen, num_iteration=best_iteration)

    validation_mse = float(mean_squared_error(y_validation, val_pred))
    unseen_pearson = safe_pearson(y_unseen, unseen_pred)
    unseen_spearman = safe_spearman(y_unseen, unseen_pred)

    trial.set_user_attr("best_iteration", int(best_iteration))
    trial_records.append({
        "trial": trial.number,
        "validation_mse": validation_mse,
        "unseen_pearson": unseen_pearson,
        "unseen_spearman": unseen_spearman,
    })
    trial_models[trial.number] = model

    print(
        f"Trial {trial.number:3d} | Validation MSE: {validation_mse:.6f} | "
        f"Unseen Pearson: {unseen_pearson:.4f} | "
        f"Unseen Spearman: {unseen_spearman:.4f}"
    )
    return validation_mse

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
)
study.optimize(objective, n_trials=N_TRIALS)


[I 2026-07-15 21:30:42,186] A new study created in memory with name: no-name-38fee443-c59a-48ac-8846-4dae2ccef51f
[I 2026-07-15 21:30:57,897] Trial 0 finished with value: 0.7408824936866997 and parameters: {'num_leaves': 101, 'min_child_samples': 244}. Best is trial 0 with value: 0.7408824936866997.


Trial   0 | Validation MSE: 0.740882 | Unseen Pearson: 0.5050 | Unseen Spearman: 0.4721


[I 2026-07-15 21:31:15,794] Trial 1 finished with value: 0.7411326413576494 and parameters: {'num_leaves': 190, 'min_child_samples': 157}. Best is trial 0 with value: 0.7408824936866997.


Trial   1 | Validation MSE: 0.741133 | Unseen Pearson: 0.5034 | Unseen Spearman: 0.4677


[I 2026-07-15 21:31:32,505] Trial 2 finished with value: 0.7400234149758733 and parameters: {'num_leaves': 46, 'min_child_samples': 46}. Best is trial 2 with value: 0.7400234149758733.


...

Trial  97 | Validation MSE: 0.739490 | Unseen Pearson: 0.5039 | Unseen Spearman: 0.4683


[I 2026-07-15 21:56:39,671] Trial 98 finished with value: 0.7363328498294943 and parameters: {'num_leaves': 34, 'min_child_samples': 174}. Best is trial 31 with value: 0.7297199787020581.


Trial  98 | Validation MSE: 0.736333 | Unseen Pearson: 0.5060 | Unseen Spearman: 0.4712


[I 2026-07-15 21:56:51,084] Trial 99 finished with value: 0.7338446588042903 and parameters: {'num_leaves': 23, 'min_child_samples': 84}. Best is trial 31 with value: 0.7297199787020581.


Trial  99 | Validation MSE: 0.733845 | Unseen Pearson: 0.5029 | Unseen Spearman: 0.4647


In [13]:
results_df = pd.DataFrame(trial_records).sort_values("trial").reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / "all_100_trial_metrics.csv", index=False)
results_df["validation_mse"].to_csv(OUTPUT_DIR / "Validation_loss.txt", index=False, header=False)
results_df["unseen_pearson"].to_csv(OUTPUT_DIR / "Unseen_Pearson.txt", index=False, header=False)
results_df["unseen_spearman"].to_csv(OUTPUT_DIR / "Unseen_Spearman.txt", index=False, header=False)

best_trial = study.best_trial.number
best_row = results_df.loc[results_df["trial"] == best_trial].iloc[0]
best_model = trial_models[best_trial]

joblib.dump({
    "model": best_model,
    "tracr": TRACR_FILTER,
    "feature_columns": X_train.columns.tolist(),
    "best_trial": best_trial,
    "best_params": study.best_trial.params,
    "best_iteration": study.best_trial.user_attrs["best_iteration"],
    "validation_mse": float(best_row["validation_mse"]),
    "unseen_pearson": float(best_row["unseen_pearson"]),
    "unseen_spearman": float(best_row["unseen_spearman"]),
}, OUTPUT_DIR / "best_model.joblib")

with open(OUTPUT_DIR / "best_trial_summary.json", "w") as f:
    json.dump({
        "tracr": TRACR_FILTER,
        "best_trial": int(best_trial),
        "best_params": study.best_trial.params,
        "best_iteration": int(study.best_trial.user_attrs["best_iteration"]),
        "validation_mse": float(best_row["validation_mse"]),
        "unseen_pearson": float(best_row["unseen_pearson"]),
        "unseen_spearman": float(best_row["unseen_spearman"]),
    }, f, indent=2)

display(results_df.head())
print("\nBest trial:", best_trial)
print(best_row[["validation_mse", "unseen_pearson", "unseen_spearman"]])
print("Saved to:", OUTPUT_DIR.resolve())


,trial,validation_mse,unseen_pearson,unseen_spearman
0,0,0.740882,0.505017,0.472079
1,1,0.741133,0.503406,0.467731
2,2,0.740023,0.510556,0.470870
3,3,0.741087,0.503310,0.469415
4,4,0.741985,0.505215,0.471704



Best trial: 31
validation_mse     0.729720
unseen_pearson     0.507080
unseen_spearman    0.468985
Name: 31, dtype: float64
Saved to: /Users/zhangjiongyu/CRISPR-modular-deep-learning_backup_copy_revision/Revision_existing_model_comparison/RS3/rs_dev/models/rs3_chen2013_fixed_split_100_trials
